# Transform refunds data
1. Extract specific portion of the string from refund_reason using split function
2. Extract specific portion of the string from refund_reason using regexp_extraction function
3. Extract data and timd from the refund_timestamp
4. Write transformed data to the silver schema

####1. Extract specific portion of the string from refund_reason using split function


In [0]:
%sql

REFRESH TABLE gizmobox.bronze.refunds;

SELECT
refund_id,
payment_id,
refund_timestamp,
refund_amount,
refund_reason 
FROM gizmobox.bronze.refunds LIMIT 5;

refund_id,payment_id,refund_timestamp,refund_amount,refund_reason
1,66,2025-01-10T11:30:00Z,85.75,Payment Error:Retailer
2,69,2025-01-03T12:40:15Z,120.50,Order Cancelled:Customer
3,72,2025-01-06T14:45:30Z,65.00,Product Returned:Customer
4,73,2025-01-07T16:10:45Z,210.99,Order Cancelled:Customer
5,75,2025-01-09T18:25:00Z,45.20,Payment Error:Retailer


####2. Extract specific portion of the string from refund_reason using regexp_extraction function


In [0]:
%sql
SELECT
refund_id,
payment_id,
refund_timestamp,
refund_amount,
SPLIT(refund_reason, ":") 
FROM gizmobox.bronze.refunds LIMIT 5;

refund_id,payment_id,refund_timestamp,refund_amount,"split(refund_reason, :, -1)"
1,66,2025-01-10T11:30:00Z,85.75,"List(Payment Error, Retailer)"
2,69,2025-01-03T12:40:15Z,120.50,"List(Order Cancelled, Customer )"
3,72,2025-01-06T14:45:30Z,65.00,"List(Product Returned, Customer )"
4,73,2025-01-07T16:10:45Z,210.99,"List(Order Cancelled, Customer )"
5,75,2025-01-09T18:25:00Z,45.20,"List(Payment Error, Retailer)"


In [0]:
%sql
SELECT
refund_id,
payment_id,
refund_timestamp,
refund_amount,
SPLIT(refund_reason, ":")[0] AS refund_reason,
SPLIT(refund_reason, ":")[1] AS refund_source
FROM gizmobox.bronze.refunds;

refund_id,payment_id,refund_timestamp,refund_amount,refund_reason,refund_source
1,66,2025-01-10T11:30:00Z,85.75,Payment Error,Retailer
2,69,2025-01-03T12:40:15Z,120.50,Order Cancelled,Customer
3,72,2025-01-06T14:45:30Z,65.00,Product Returned,Customer
4,73,2025-01-07T16:10:45Z,210.99,Order Cancelled,Customer
5,75,2025-01-09T18:25:00Z,45.20,Payment Error,Retailer
6,80,2025-01-10T09:35:20Z,130.15,Order Cancelled,Customer
7,83,2025-01-12T11:20:40Z,150.00,Product Returned,Customer
8,85,2025-01-14T13:15:30Z,89.99,Order Cancelled,Customer
9,89,2025-01-15T15:00:00Z,78.50,Payment Error,Retailer
10,91,2025-01-17T16:45:15Z,250.75,Product Returned,Customer


####3. Extract data and timd from the refund_timestamp

In [0]:
%sql
SELECT
refund_id,
payment_id,
CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
date_format(refund_timestamp, 'HH-mm-ss') AS refund_time,
refund_amount,
SPLIT(refund_reason, ":")[0] AS refund_reason,
SPLIT(refund_reason, ":")[1] AS refund_source
FROM gizmobox.bronze.refunds;

refund_id,payment_id,refund_date,refund_time,refund_amount,refund_reason,refund_source
1,66,2025-01-10,11-30-00,85.75,Payment Error,Retailer
2,69,2025-01-03,12-40-15,120.50,Order Cancelled,Customer
3,72,2025-01-06,14-45-30,65.00,Product Returned,Customer
4,73,2025-01-07,16-10-45,210.99,Order Cancelled,Customer
5,75,2025-01-09,18-25-00,45.20,Payment Error,Retailer
6,80,2025-01-10,09-35-20,130.15,Order Cancelled,Customer
7,83,2025-01-12,11-20-40,150.00,Product Returned,Customer
8,85,2025-01-14,13-15-30,89.99,Order Cancelled,Customer
9,89,2025-01-15,15-00-00,78.50,Payment Error,Retailer
10,91,2025-01-17,16-45-15,250.75,Product Returned,Customer


####4. Write transformed data to the silver schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.refunds
SELECT
refund_id,
payment_id,
CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
date_format(refund_timestamp, 'HH-mm-ss') AS refund_time,
refund_amount,
SPLIT(refund_reason, ":")[0] AS refund_reason,
SPLIT(refund_reason, ":")[1] AS refund_source
FROM gizmobox.bronze.refunds;

num_affected_rows,num_inserted_rows
